In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
# Importing Libraries

import os
import numpy as np
import pandas as pd
import cv2
import random
import re

from glob import glob
import time
import json
import gc

import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, auc, log_loss

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import Sequential
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping, ModelCheckpoint
from tensorflow.keras.applications import VGG16, ResNet50, EfficientNetB0, EfficientNetB1, EfficientNetB2

# for "Clear TensorFlow/Keras session before building a new model"
from tensorflow.keras import mixed_precision
from tensorflow.keras import backend as K

In [ ]:
# GPU error prevention and detection of GPU usage

# - Enable memory growth
for gpu in tf.config.list_physical_devices('GPU'):
    tf.config.experimental.set_memory_growth(gpu, True)

# - GPU or CPU is using
gpu_devices = tf.config.list_physical_devices('GPU')
if len(gpu_devices) > 1:
    print('✅ Two GPUs are using') 
elif len(gpu_devices) == 1:
    print('✅ GPU is using') 
else: 
    print('⚠️ CPU is using')


In [ ]:
# 2. Load Dataset
    # - From: "/kaggle/input/kidney-stone-classification-and-object-detection"

data_dir = "/kaggle/input/datasets/imtkaggleteam/kidney-stone-classification-and-object-detection"
class_names = os.listdir(data_dir)
print("Classes:", class_names)

for class_name in class_names:
    folder_path = os.path.join(data_dir, class_name)
    print(f"{class_name}: {len(os.listdir(folder_path))} images")

In [ ]:
# Upload photos separately "normal" and "stone"

image_list_stone = sorted(glob(data_dir + "/stone/*.JPG"))
image_list_normal = sorted(glob(data_dir + "/Normal/*.JPG"))

In [ ]:
len(image_list_stone)

In [ ]:
len(image_list_normal)

In [ ]:
# Create DataFrames
df_normal = pd.DataFrame({'image_path': image_list_normal, 'label': 0})
df_stone = pd.DataFrame({'image_path': image_list_stone, 'label': 1})

df_all = pd.concat([df_normal, df_stone], ignore_index=True)
df_all = df_all.sample(frac=1, random_state=42).reset_index(drop=True)

df = df_all
df.head()


**Data Exploration & Preprocessing**

In [ ]:
print(df.info())

In [ ]:
print(df.isnull().sum())

In [ ]:
print(df.duplicated().sum())

**Sample Images and Class Distribution**

In [ ]:
# Show some sample images from each class

def show_sample_images(data_dir, class_names, img_per_class=4):
    plt.figure(figsize=(12, 6))
    for i, class_name in enumerate(class_names):
        img_list = os.listdir(os.path.join(data_dir, class_name))
        for j in range(img_per_class):
            img_path = os.path.join(data_dir, class_name, img_list[j])
            img = cv2.imread(img_path)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            plt.subplot(len(class_names), img_per_class, i * img_per_class + j + 1)
            plt.imshow(img)
            plt.axis('off')
            plt.title(class_name)
    plt.tight_layout()
    plt.show()

show_sample_images(data_dir, class_names)

In [ ]:
#Convert numerical labels to string format for ImageDataGenerator
df_all['label'] = df_all['label'].map({0: 'Normal', 1: 'Stone'})


In [ ]:
#Display 5 random images from the dataframe
for i in range(5):
    rand_idx = random.randint(0, len(df) - 1)
    img_path = df.iloc[rand_idx]['image_path']  
    label = df.iloc[rand_idx]['label']
    
    img = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    print(f"Label: {label}")
    plt.imshow(img_rgb)
    plt.title(f"Sample {i+1} - Label: {label}")
    plt.axis('off')
    plt.show()

    # We keep the last image for dimension checking.
    if i == 4:
        last_img = img

# Check the dimensions of the last image
height, width = last_img.shape[:2]
print("🔍 Dimensions of the last image:")
print("The height is:", height)
print("The width is:", width)

In [ ]:
df['label'].value_counts(normalize=True)


In [ ]:
sns.set_style('whitegrid')  # or "darkgrid"

colors = ['#CB0404', '#309898']  

counts = {cls: len(os.listdir(os.path.join(data_dir, cls))) for cls in class_names}

# Drawing a bar chart
plt.figure(figsize=(6, 4))
sns.barplot(
    x=list(counts.keys()),
    y=list(counts.values()),
    hue=list(counts.keys()),
    palette=colors
)

plt.title("Class Distribution", fontsize=14, weight='bold')
plt.xlabel("Class", fontsize=12)
plt.ylabel("Number of Images", fontsize=12)
plt.xticks(fontsize=10)
plt.yticks(fontsize=10)
plt.tight_layout()
plt.show()


**Data Preprocessing**

In [ ]:
BATCH_SIZE = 32

IMG_HEIGHT, IMG_WIDTH = 96, 96

INPUT_SHAPE = (IMG_HEIGHT, IMG_WIDTH, 3) # RGB image input (input 3D)

SEED = 42

EPOCHS = 20 # Changed from "EPOCHS = 10 and 15"

NUM_CLASSES = 2

In [ ]:
# Create initial DataFrames with string labels
df_normal = pd.DataFrame({'image_path': image_list_normal, 'label': 'Normal'})
df_stone  = pd.DataFrame({'image_path': image_list_stone,  'label': 'Stone'})

In [ ]:
df_normal.head()

In [ ]:
# Split each class separately

train_ratio, val_ratio, test_ratio = 0.7, 0.2, 0.1

NUM_GPUS = len(tf.config.list_physical_devices('GPU'))
if NUM_GPUS == 0: NUM_GPUS += 1
EFFECTIVE_BATCH = BATCH_SIZE * NUM_GPUS

def classwise_split(image_list, label):
    total = len(image_list)
    train_count = int((train_ratio * total) // EFFECTIVE_BATCH * EFFECTIVE_BATCH)
    val_count   = int((val_ratio * total)  // EFFECTIVE_BATCH * EFFECTIVE_BATCH)
    test_count  = int((test_ratio * total) // EFFECTIVE_BATCH * EFFECTIVE_BATCH)

    total_count = train_count + val_count + test_count
    image_list = random.sample(image_list, total_count)

    train = image_list[:train_count]
    val   = image_list[train_count:train_count + val_count]
    test  = image_list[train_count + val_count:]

    df_train = pd.DataFrame({'image_path': train, 'label': label})
    df_val   = pd.DataFrame({'image_path': val,   'label': label})
    df_test  = pd.DataFrame({'image_path': test,  'label': label})

    return df_train, df_val, df_test

In [ ]:
df_train_normal, df_val_normal, df_test_normal = classwise_split(image_list_normal, 'Normal')
df_train_stone,  df_val_stone,  df_test_stone  = classwise_split(image_list_stone,  'Stone')

In [ ]:
# Combine, Shuffle and Report
df_train = pd.concat([df_train_normal, df_train_stone]).sample(frac=1, random_state=SEED).reset_index(drop=True)
df_val   = pd.concat([df_val_normal, df_val_stone]).sample(frac=1, random_state=SEED).reset_index(drop=True)
df_test  = pd.concat([df_test_normal, df_test_stone]).sample(frac=1, random_state=SEED).reset_index(drop=True)

print(f"Total training samples: {len(df_train)}")
print(f"Total validation samples: {len(df_val)}")
print(f"Total testing samples: {len(df_test)}")

In [ ]:
df_train.head(10)

In [ ]:
df_test.head(10)

In [ ]:
# Data Augmentation Setup
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True
)

val_test_datagen = ImageDataGenerator(rescale=1./255)

In [ ]:
# Load Data Using flow_from_dataframe

trainDataset = train_datagen.flow_from_dataframe(
    dataframe=df_train,
    x_col="image_path",
    y_col="label",
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,  # ❌ batch_size=1 for trainDataset
    class_mode="categorical",
    shuffle=True,
    seed=SEED
)

valDataset = val_test_datagen.flow_from_dataframe(
    dataframe=df_val,
    x_col="image_path",
    y_col="label",
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

testDataset = val_test_datagen.flow_from_dataframe(
    dataframe=df_test,
    x_col="image_path",
    y_col=None,  # <--- No label column
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode=None,  # <--- No class mode
    shuffle=False  # <--- So prediction order aligns with dataframe
)

In [ ]:
# Generate labeled test data to evaluate the model's performance
testDataset_with_labels = val_test_datagen.flow_from_dataframe(
    dataframe=df_test,
    x_col="image_path",
    y_col="label",  # label column  in testDataset
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode="categorical",  # one-hot
    shuffle=False 
)

In [ ]:
# A function that converts the Generator to a tf.data.Dataset [Usable for training]

def dataset_from_generator(generator, has_labels=True):
    if has_labels:
        return tf.data.Dataset.from_generator(
            lambda: generator,
            output_signature=(
                tf.TensorSpec(shape=(BATCH_SIZE, IMG_HEIGHT, IMG_WIDTH, 3), dtype=tf.float32),
                tf.TensorSpec(shape=(BATCH_SIZE, NUM_CLASSES), dtype=tf.float32)
            )
        ).prefetch(tf.data.AUTOTUNE)  # ✅ no ".batch()"!
    else:
        return tf.data.Dataset.from_generator(
            lambda: generator,
            output_signature=tf.TensorSpec(shape=(BATCH_SIZE, IMG_HEIGHT, IMG_WIDTH, 3), dtype=tf.float32)
        ).prefetch(tf.data.AUTOTUNE)

In [ ]:
# Set precision (in case of NaN problem it is better to use float32)

mixed_precision.set_global_policy("float32")

In [ ]:
# Strategy setup (once): Define MirroredStrategy only once, outside the `train_model` function

gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    strategy = tf.distribute.MirroredStrategy()
    print(f"✅ GPUs detected: {[gpu.name for gpu in gpus]}")
else:
    print("⚠ No GPU detected! Running on CPU.")
    strategy = tf.distribute.OneDeviceStrategy(device="/CPU:0")

In [ ]:
# - Save function
def save_training_history(history_file, full_acc, full_val_acc, full_loss, full_val_loss):
    # history_file = f"{model_name}_history.npz"
    try:
        np.savez(history_file, acc=full_acc, val_acc=full_val_acc, loss=full_loss, val_loss=full_val_loss)
        print("✅ Training history saved!")
    except Exception as e:
        print(f"⚠️ Error saving history: {e}")

# - Load function
def load_training_history(history_file):
    # history_file = f"{model_name}_history.npz"
    if os.path.exists(history_file):
        try:
            data = np.load(history_file)
            print(">>> Previous training history loaded.")
            return (
                list(data['acc']) if 'acc' in data else [],
                list(data['val_acc']) if 'val_acc' in data else [],
                list(data['loss']) if 'loss' in data else [],
                list(data['val_loss']) if 'val_loss' in data else []
            )
        except Exception as e:
            print(f"⚠️ Error loading history: {e}")
            return [], [], [], []
    else:
        print("*** First training run, starting from scratch.")
        return [], [], [], []

In [ ]:
# 📌 Function to save model predictions with `re.sub` (both labels + softmax probs)

def save_test_predictions(df_test, test_steps, categorical_predictions, probabilities, model_name, class_labels):
    """
    Save test predictions including:
      - Image IDs
      - Predicted label (hard classification)
      - Softmax probabilities for each class
    """

    # 🔹 Helper to clean non-ASCII chars from file paths
    def clean_string(s):
        if not isinstance(s, str):
            s = str(s)
        return re.sub(r'[^\x00-\x7F]+', '', s)

    # Define number of test samples (truncate to full batches)
    valid_test_count = test_steps * BATCH_SIZE

    # Extract and clean IDs
    ids = df_test['image_path'].values[:valid_test_count]
    cleaned_ids = [clean_string(x) for x in ids]

    # Create DataFrame with hard labels + softmax probabilities
    output_df = pd.DataFrame({
        'id': cleaned_ids,
        'label': categorical_predictions.astype(int)   # 0/1 hard label
    })

    # Append probability columns
    for i, cls in enumerate(class_labels):
        output_df[f"prob_{cls}"] = probabilities[:, i]

    # Save to CSV
    output_filename = f"{model_name.replace('.keras', '')}_predictions.csv"
    output_df.to_csv(output_filename, index=False)
    print(f"📁 Predictions saved to '{output_filename}'")

In [ ]:
def save_model_metrics(model_name, y_true, y_pred, results_file="models_evaluation_results.csv"):
    from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, log_loss

    # Calculate metrics
    acc = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    try:
        loss = log_loss(y_true, y_pred)
    except:
        loss = None  # In case of error (e.g. binary hard labels)

    # Create results row
    result_row = {
        "Model": model_name,
        "Accuracy": acc,
        "Precision": precision,
        "Recall": recall,
        "F1-score": f1,
        "LogLoss": loss
    }

    # If file exists, append, else create new
    if os.path.exists(results_file):
        df = pd.read_csv(results_file)
        df = pd.concat([df, pd.DataFrame([result_row])], ignore_index=True)
    else:
        df = pd.DataFrame([result_row])

    # Save back
    df.to_csv(results_file, index=False)
    print(f"✅ Results for {model_name} saved to {results_file}")

In [ ]:
# 📁 Directory to save

OUTPUT_DIR = "results/"
os.makedirs(OUTPUT_DIR, exist_ok=True)


# 🧩 Function: Save predictions (CSV + JSON)

def save_final_predictions_for_task2(
    df_test, test_steps, categorical_predictions, probabilities, model_name, class_labels, batch_size=BATCH_SIZE
):
    """
    Save model predictions for Task 2 linkage:
      - CSV: 'classification_results.csv'
      - JSON: 'classification_results.json'
    Includes image names, predicted labels, and confidence scores.
    """

    def clean_string(s):
        if not isinstance(s, str):
            s = str(s)
        return re.sub(r"[^\x00-\x7F]+", "", s)

    valid_test_count = test_steps * batch_size
    ids = df_test["image_path"].values[:valid_test_count]
    cleaned_ids = [os.path.basename(clean_string(x)) for x in ids]

    # Extract hard predictions and softmax probs
    hard_labels = categorical_predictions.astype(int)
    confidences = np.max(probabilities, axis=1)

    output_df = pd.DataFrame({
        "file_name": cleaned_ids,
        "label": hard_labels,
        "confidence": confidences
    })

    # 🔹 Save CSV
    csv_path = os.path.join(OUTPUT_DIR, "classification_results.csv")
    output_df.to_csv(csv_path, index=False)
    print(f"✅ CSV saved: {csv_path}")

    # 🔹 Save JSON
    json_data = [
        {"file_name": row["file_name"],
         "label": "stone" if row["label"] == 1 else "normal",
         "confidence": float(row["confidence"])}
        for _, row in output_df.iterrows()
    ]
    json_path = os.path.join(OUTPUT_DIR, "classification_results.json")
    with open(json_path, "w") as jf:
        json.dump(json_data, jf, indent=4)
    print(f"✅ JSON saved: {json_path}")

    # Return DataFrame for later use (filtering Task 2 inputs)
    return output_df

In [ ]:
# History loading function and visualization "Accuracy" plot

def plot_accuracy(model_name):
    history_file = f"{model_name}_history.npz"
    if not os.path.exists(history_file): return print("⚠️ No history found.")
    data = np.load(history_file)
    plt.figure(figsize=(8, 6))
    plt.plot(data['acc'], label='Train Accuracy')
    plt.plot(data['val_acc'], label='Val Accuracy')
    plt.xlabel('Epochs') 
    plt.ylabel('Accuracy') 
    plt.title(f"{model_name} Accuracy")
    plt.legend()
    plt.grid() 
    plt.show()


# History loading function and visualization "Loss" plot

def plot_loss(model_name):
    history_file = f"{model_name}_history.npz"
    if not os.path.exists(history_file): return print("⚠️ No history found.")
    data = np.load(history_file)
    plt.figure(figsize=(8, 6))
    plt.plot(data['loss'], label='Train Loss')
    plt.plot(data['val_loss'], label='Val Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss') 
    plt.title(f"{model_name} Loss")
    plt.legend() 
    plt.grid()
    plt.show()

In [ ]:
# Function to compare the "Accuracy" of multiple models together

def compare_models_accuracy(models):
    plt.figure(figsize=(10, 6))
    for name in models:
        path = f"{name}_history.npz"
        if os.path.exists(path):
            data = np.load(path)
            plt.plot(data['acc'], label=f"{name} - Train")
            plt.plot(data['val_acc'], '--', label=f"{name} - Val")
    plt.title("Model Accuracy Comparison")
    plt.xlabel("Epochs") 
    plt.ylabel("Accuracy")
    plt.legend() 
    plt.grid() 
    plt.show()


# Function to compare the "Loss" of multiple models together

def compare_models_loss(models):
    plt.figure(figsize=(10, 6))
    for name in models:
        path = f"{name}_history.npz"
        if os.path.exists(path):
            data = np.load(path)
            plt.plot(data['loss'], label=f"{name} - Train")
            plt.plot(data['val_loss'], '--', label=f"{name} - Val")
    plt.title("Model Loss Comparison")
    plt.xlabel("Epochs")
    plt.ylabel("Loss")
    plt.legend() 
    plt.grid() 
    plt.show()

In [ ]:
# 📌 Training Strategy & Model Management (Main training pipeline)
#   - Using `tf.data.Dataset` for trainDataset (trainDataset_tf)
#   - Modified to save both hard labels & softmax probs

def train_model(model_name, model_fn, trainDataset, valDataset, testDataset, epochs, class_labels):
    # 0) Clear session
    K.clear_session()
    gc.collect()
    tf.compat.v1.reset_default_graph()

    start_time = time.time()

    # 1) Load previous history
    history_file = f"{model_name}_history.npz"
    prev_acc, prev_val_acc, prev_loss, prev_val_loss = load_training_history(history_file)

    # 2) Build or load model
    with strategy.scope():
        if os.path.exists(model_name):
            print(f">>> Loading existing model from '{model_name}'...")
            model = tf.keras.models.load_model(model_name)
        else:
            print("*** Building new model...")
            model = model_fn()
            model.compile(
                loss='categorical_crossentropy',
                optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
                metrics=['accuracy']
            )

    if not os.path.exists(model_name):
        model.summary()

    print(f"[INFO] Dataset sizes -> Train: {len(trainDataset)}, Val: {len(valDataset)}, Test: {len(testDataset)}")
    assert len(trainDataset) % strategy.num_replicas_in_sync == 0, "Training set not divisible by number of GPUs"

    # 3) Convert generators to tf.data
    trainDataset_tf = dataset_from_generator(trainDataset, has_labels=True)
    valDataset_tf   = dataset_from_generator(valDataset, has_labels=True)
    testDataset_tf  = dataset_from_generator(testDataset, has_labels=False)

    # 4) Callbacks
    reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6, verbose=1)
    early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=1)
    checkpoint = ModelCheckpoint(model_name, monitor='val_loss', save_best_only=True, mode='min', verbose=1)
    callbacks = [reduce_lr, early_stopping, checkpoint]

    # 5) Inspect batch
    for sample_batch in trainDataset_tf.take(1):
        if isinstance(sample_batch, tuple):
            print("Batch shape (X):", sample_batch[0].shape)
            print("Batch shape (Y):", sample_batch[1].shape)

    # 6) Fit model
    steps_per_epoch = len(trainDataset) 
    validation_steps = len(valDataset)
    test_steps = df_test.shape[0] // BATCH_SIZE

    history = model.fit(
        trainDataset_tf,
        validation_data=valDataset_tf,
        callbacks=callbacks,
        steps_per_epoch=steps_per_epoch,
        validation_steps=validation_steps,
        epochs=epochs
    )

    # Save training meta info
    train_time_sec = time.time() - start_time
    meta_info = {
        "train_time_sec": train_time_sec,
        "params": {
            "batch_size": BATCH_SIZE,
            "learning_rate": 1e-4,
            "epochs": epochs
        }
    }
    with open(model_name.replace(".keras", "_meta.json"), "w") as f:
        json.dump(meta_info, f, indent=4)

    # 7) Save history
    full_acc = prev_acc + history.history['accuracy']
    full_val_acc = prev_val_acc + history.history['val_accuracy']
    full_loss = prev_loss + history.history['loss']
    full_val_loss = prev_val_loss + history.history['val_loss']
    save_training_history(history_file, full_acc, full_val_acc, full_loss, full_val_loss)

    # 8) Load best model
    if os.path.exists(model_name):
        print(f"✅ Model '{model_name}' loaded.")
        best_model = tf.keras.models.load_model(model_name)
    else:
        print(f"❌ Model '{model_name}' not found!")
        return

    print("Evaluating on validation dataset...")
    loss, acc = best_model.evaluate(valDataset, verbose=1, steps=validation_steps)
    print(f"✅ Validation Loss: {loss:.4f}, Accuracy: {acc:.4f}")

    # 9) Predict on test set
    print("Predicting on test dataset...")
    # predictions: softmax probabilities, shape (N, num_classes)
    predictions = best_model.predict(testDataset, verbose=1, steps=test_steps)
    # hard labels from softmax
    categorical_predictions = np.argmax(predictions, axis=1)

    # 10) Save predictions (labels + probs)  -- per-model CSV (existing function)
    save_test_predictions(df_test, test_steps, categorical_predictions, predictions, model_name, class_labels)

    # 11) Save final predictions for Task2 (CSV + JSON)
    final_df = save_final_predictions_for_task2(
        df_test=df_test,
        test_steps=test_steps,
        categorical_predictions=categorical_predictions,
        probabilities=predictions,         # <-- use 'predictions' here
        model_name=model_name,
        class_labels=class_labels,
        batch_size=BATCH_SIZE
    )

    ##return best_model
    
    # Optional: return both model and dataframe for later use
    return best_model, final_df

In [ ]:
'''
# Training Strategy & Model Management
    # Using basic `flow_from_dataframe` for training data (trainDataset)

def train_model(model_name, model_fn, trainDataset, valDataset, testDataset, epochs):
    # 0) Clear previous session to avoid residual graphs or shape mismatches
        # - Clear previous Keras/TensorFlow session to avoid leftover graphs or memory leaks
        # - Ensures clean state when training multiple models in a single run
    K.clear_session()
    gc.collect()
    tf.compat.v1.reset_default_graph()

    # 1) Load previous training history (if available)
        # - Load training history if a previous run exists, allowing resume or graph plotting
        # - Useful especially with early stopping or interrupted training
    history_file = f"{model_name}_history.npz"
    prev_acc, prev_val_acc, prev_loss, prev_val_loss = load_training_history(history_file)

    # 2) Build or load the model within distribution strategy
        # - If the model exists on disk, load it; otherwise, build and compile a new model
        # - `strategy.scope()` enables distributed training on multiple GPUs (MirroredStrategy)
    with strategy.scope():
        if os.path.exists(model_name):
            print(f">>> Loading existing model from '{model_name}'...")
            model = tf.keras.models.load_model(model_name)
        else:
            print("*** Building new model...")
            model = model_fn()
            optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3, clipnorm=1.0)
            model.compile(loss='categorical_crossentropy', optimizer=optimizer, metrics=['accuracy'])

    # Print the model structure if it's freshly built
    if not os.path.exists(model_name):
        model.summary()

    # Print dataset sizes and assert compatibility with distributed training
    print(f"[INFO] Dataset sizes -> Train: {len(trainDataset)}, Val: {len(valDataset)}, Test: {len(testDataset)}")
    assert len(trainDataset) % strategy.num_replicas_in_sync == 0, "Training set not divisible by number of GPUs"

    # 3) Wrap ImageDataGenerators into optimized tf.data.Datasets 
        # - Convert Keras generators into tf.data.Dataset objects [Because it performs better with MirroredStrategy]
        # - Enables prefetching and compatibility with distributed GPU training

    ##trainDataset_tf = dataset_from_generator(trainDataset, has_labels=True).prefetch(tf.data.AUTOTUNE)
    ##valDataset_tf   = dataset_from_generator(valDataset, has_labels=True).prefetch(tf.data.AUTOTUNE)
    ##testDataset_tf  = dataset_from_generator(testDataset, has_labels=False).prefetch(tf.data.AUTOTUNE)

    # 4) Setup callbacks: ReduceLROnPlateau, EarlyStopping, and ModelCheckpoint
        # - Callbacks to enhance training:
            # 1. ReduceLROnPlateau: lowers LR on plateau in validation loss
            # 2. EarlyStopping: stops training if validation loss stagnates
            # 3. ModelCheckpoint: saves only the best performing model weights
    reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6, verbose=1)
    early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=1)
    checkpoint = ModelCheckpoint(model_name, monitor='val_loss', save_best_only=True, mode='min', verbose=1)
    callbacks = [reduce_lr, early_stopping, checkpoint]

    # 5) Inspect input batch shape to verify correctness
        # - Check shape from ImageDataGenerator directly
    sample_batch = next(trainDataset)
    if isinstance(sample_batch, tuple):
        print("Batch shape (X):", sample_batch[0].shape)
        print("Batch shape (Y):", sample_batch[1].shape)
    else:
        print("Batch shape:", sample_batch.shape)

    # 6) Train the model using fit()
        # - Train the model using the preprocessed dataset and callbacks
    steps_per_epoch = len(trainDataset)
    validation_steps = len(valDataset)
    test_steps = df_test.shape[0] // BATCH_SIZE
    valid_test_count = test_steps * BATCH_SIZE

    history = model.fit(
        trainDataset,
        validation_data=valDataset,
        callbacks=callbacks,
        steps_per_epoch=steps_per_epoch,
        validation_steps=validation_steps,
        epochs=epochs
    )

    # 7) Save full training history
        # - Combine current history with previously loaded history
        # - Save cumulative training metrics to an .npz file
    full_acc = prev_acc + history.history['accuracy']
    full_val_acc = prev_val_acc + history.history['val_accuracy']
    full_loss = prev_loss + history.history['loss']
    full_val_loss = prev_val_loss + history.history['val_loss']

    save_training_history(history_file, full_acc, full_val_acc, full_loss, full_val_loss)

    # 8) Load best model and evaluate
        # - Reload best weights saved during training
        # - Evaluate final performance on validation set
    if os.path.exists(model_name):
        print(f"\u2705 Model '{model_name}' loaded.")
        best_model = tf.keras.models.load_model(model_name)
    else:
        print(f"\u274c Model '{model_name}' not found!")
        return

    print("Evaluating on validation dataset...")
    loss, acc = best_model.evaluate(valDataset, verbose=1, steps=validation_steps)
    print(f"\u2705 Validation Loss: {loss:.4f}, Accuracy: {acc:.4f}")

    # 9) Predict test set
        # - Perform predictions on the test set using best saved weights
        # - Convert softmax probabilities
    print("Predicting on test dataset...")
    predictions = best_model.predict(testDataset, verbose=1, steps=test_steps)
    categorical_predictions = np.argmax(predictions, axis=1) # ّbecause it is "Softmax"
    ## binary_predictions = (predictions > 0.5).astype(int).flatten() # If it were "Sigmoid"

    # 10) Save test predictions
        # - Store predictions along with sample IDs into a CSV file
        # - Can be used for leaderboard submission or post-analysis
    save_test_predictions(df_test, test_steps, categorical_predictions, model_name)


    return best_model

'''

In [ ]:
# 🔰 simple_cnn_model_fn: The simplest CNN base model for initial sanity check
def simple_cnn_model_fn(input_shape=(96, 96, 3), num_classes=2):
    """
    Build a simple CNN model for image classification.
    
    Parameters:
        input_shape (tuple): Shape of the input images (default: 96x96 RGB).
        num_classes (int): Number of output classes (default: 2).
        
    Returns:
        model (tf.keras.Model): Compiled CNN model.
    """

    model = models.Sequential([
        layers.Input(shape=input_shape),

        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),

        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')  # Softmax for categorical output
    ])

    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',  # Suitable for one-hot categorical labels
        metrics=['accuracy']
    )

    return model

In [ ]:
# Visual inspection of one batch from train dataset
x_batch, y_batch = next(iter(trainDataset))

print("Sample labels:", y_batch[:5])
print("Any NaNs in X?", np.isnan(x_batch).any())
print("Any NaNs in Y?", np.isnan(y_batch).any())
print("Sum of Y labels:", np.sum(y_batch, axis=1))  # should all be 1.0 in one-hot

In [ ]:
# 🔧 Set model name and epochs for training
model_name = "simple_cnn_model.keras"
epochs = 10  # Small number for sanity check
class_labels = ["Normal", "Stone"]

# 🚀 Train the simple CNN model using defined training strategy
best_model = train_model(
    model_name=model_name,
    model_fn=simple_cnn_model_fn,
    trainDataset=trainDataset,
    valDataset=valDataset,
    testDataset=testDataset,
    epochs=epochs,
    class_labels = class_labels
)

In [ ]:
# 📊 Plot Accuracy of the simple CNN model
plot_accuracy("simple_cnn_model.keras")

# 📉 Plot Loss of the simple CNN model
plot_loss("simple_cnn_model.keras")

In [ ]:
# 📦 model_fn: Building different CNN models based on the chosen architecture name
def model_fn(model_name, input_shape=(96, 96, 3), num_classes=2):
    
    if model_name == "AlexNet":
        # 📌 AlexNet-inspired architecture
        model = models.Sequential([
            layers.Input(shape=input_shape),
            layers.Conv2D(96, kernel_size=11, strides=4, activation='relu'),
            layers.BatchNormalization(),
            layers.MaxPooling2D(pool_size=3, strides=2),

            layers.Conv2D(256, kernel_size=5, padding='same', activation='relu'),
            layers.BatchNormalization(),
            layers.MaxPooling2D(pool_size=3, strides=2),

            layers.Conv2D(384, kernel_size=3, padding='same', activation='relu'),
            layers.Conv2D(384, kernel_size=3, padding='same', activation='relu'),
            layers.Conv2D(256, kernel_size=3, padding='same', activation='relu'),
            layers.MaxPooling2D(pool_size=3, strides=2),

            layers.Flatten(),
            layers.Dense(4096, activation='relu'),
            layers.Dropout(0.5),
            layers.Dense(4096, activation='relu'),
            layers.Dropout(0.5),
            layers.Dense(num_classes, activation='softmax')
        ])

    elif model_name == "VGGNet":
        # 📌 VGGNet-style deep CNN
        # ✅ Better VGG block: use ImageNet weights, GAP (GlobalAveragePooling2D), smaller FC
        base_model = VGG16(include_top=False, weights='imagenet', input_shape=input_shape) # ❌weights=None
        base_model.trainable = False  # freeze first; can unfreeze last blocks later for fine-tuning
    
        x = layers.GlobalAveragePooling2D()(base_model.output)   # much smaller flatten
        x = layers.Dense(256, activation='relu')(x)
        x = layers.Dropout(0.5)(x)
        output = layers.Dense(num_classes, activation='softmax')(x)
    
        model = models.Model(inputs=base_model.input, outputs=output)

    # compile after model creation (outside model_fn also ok)


    elif model_name == "ResNet34":
        # 📌 ResNet-style with skip connections (simulated using ResNet50)
        base_model = ResNet50(weights="imagenet", include_top=False, input_shape=input_shape)
        base_model.trainable = False  # Freeze base model: Set False to use pre-trained weights
        x = layers.GlobalAveragePooling2D()(base_model.output)
        x = layers.Dense(256, activation='relu')(x)
        x = layers.Dropout(0.5)(x)
        output = layers.Dense(num_classes, activation='softmax')(x)
        model = models.Model(inputs=base_model.input, outputs=output)

    elif model_name == "EfficientNetB0":
        # 📌 EfficientNetB0 - lightweight and efficient
        base_model = EfficientNetB0(weights="imagenet", include_top=False, input_shape=input_shape)
        base_model.trainable = False  # Freeze base model: Set False to use pre-trained weights
        x = layers.GlobalAveragePooling2D()(base_model.output)
        x = layers.Dense(256, activation='relu')(x)
        x = layers.Dropout(0.5)(x)
        output = layers.Dense(num_classes, activation='softmax')(x)
        model = models.Model(inputs=base_model.input, outputs=output)

    elif model_name == "EfficientNetB1":
        # 📌 EfficientNetB1 - slightly larger and deeper than B0
        base_model = EfficientNetB1(weights="imagenet", include_top=False, input_shape=input_shape)
        base_model.trainable = False  # Freeze base model: Set False to use pre-trained weights
        x = layers.GlobalAveragePooling2D()(base_model.output)
        x = layers.Dense(256, activation='relu')(x)
        x = layers.Dropout(0.5)(x)
        output = layers.Dense(num_classes, activation='softmax')(x)
        model = models.Model(inputs=base_model.input, outputs=output)

    elif model_name == "EfficientNetB2":
        # 📌 EfficientNetB2 - deeper and more complex than B1
        base_model = EfficientNetB2(weights="imagenet", include_top=False, input_shape=input_shape)
        base_model.trainable = False  # Freeze base model: Set False to use pre-trained weights
        x = layers.GlobalAveragePooling2D()(base_model.output)
        x = layers.Dense(256, activation='relu')(x)
        x = layers.Dropout(0.5)(x)
        output = layers.Dense(num_classes, activation='softmax')(x)
        model = models.Model(inputs=base_model.input, outputs=output)

    else:
        raise ValueError(f"❌ Model '{model_name}' not supported.")

    # ⚙️ Compile the model
    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',  # Suitable for one-hot categorical labels
        metrics=['accuracy']
    )

    return model

In [ ]:
# 🔧 Set model name and epochs for AlexNet
model_name = "AlexNet_model.keras"
epochs = EPOCHS
class_labels = ["Normal", "Stone"]

# 🚀 Train the AlexNet model using training function
best_model = train_model(
    model_name=model_name,
    model_fn=lambda: model_fn("AlexNet", input_shape=INPUT_SHAPE, num_classes=NUM_CLASSES),
    trainDataset=trainDataset,
    valDataset=valDataset,
    testDataset=testDataset,
    epochs=epochs,
    class_labels = class_labels
)


In [ ]:
# 📊 Plot Accuracy of AlexNet
plot_accuracy("AlexNet_model.keras")

# 📉 Plot Loss of AlexNet
plot_loss("AlexNet_model.keras")

In [ ]:
# 🔧 Set model name and epochs for VGGNet
model_name = "VGGNet_model.keras"
epochs = EPOCHS
class_labels = ["Normal", "Stone"]

# 🚀 Train the VGGNet model
best_model = train_model(
    model_name=model_name,
    model_fn=lambda: model_fn("VGGNet", input_shape=INPUT_SHAPE, num_classes=NUM_CLASSES),
    trainDataset=trainDataset,
    valDataset=valDataset,
    testDataset=testDataset,
    epochs=epochs,
    class_labels = class_labels
)

In [ ]:
# 📊 Plot Accuracy of VGGNet
plot_accuracy("VGGNet_model.keras")

# 📉 Plot Loss of VGGNet
plot_loss("VGGNet_model.keras")

In [ ]:
# 🔧 Set model name and epochs for ResNet34
model_name = "ResNet34_model.keras"
epochs = EPOCHS
class_labels = ["Normal", "Stone"]

# 🚀 Train the ResNet model
best_model = train_model(
    model_name=model_name,
    model_fn=lambda: model_fn("ResNet34", input_shape=INPUT_SHAPE, num_classes=NUM_CLASSES),
    trainDataset=trainDataset,
    valDataset=valDataset,
    testDataset=testDataset,
    epochs=epochs,
    class_labels = class_labels
)

In [ ]:
# 📊 Plot Accuracy of ResNet34
plot_accuracy("ResNet34_model.keras")

# 📉 Plot Loss of ResNet34
plot_loss("ResNet34_model.keras")

In [ ]:
# 🔧 Set model name and epochs for EfficientNetB0
model_name = "EfficientNetB0_model.keras"
epochs = EPOCHS
class_labels = ["Normal", "Stone"]

# 🚀 Train the EfficientNetB0 model
best_model = train_model(
    model_name=model_name,
    model_fn=lambda: model_fn("EfficientNetB0", input_shape=INPUT_SHAPE, num_classes=NUM_CLASSES),
    trainDataset=trainDataset,
    valDataset=valDataset,
    testDataset=testDataset,
    epochs=epochs,
    class_labels = class_labels
)

In [ ]:
# 📊 Plot Accuracy of EfficientNetB0
plot_accuracy("EfficientNetB0_model.keras")

# 📉 Plot Loss of EfficientNetB0
plot_loss("EfficientNetB0_model.keras")

In [ ]:
# 🔧 Set model name and epochs for EfficientNetB1
model_name = "EfficientNetB1_model.keras"
epochs = EPOCHS
class_labels = ["Normal", "Stone"]

# 🚀 Train the EfficientNetB1 model
best_model = train_model(
    model_name=model_name,
    model_fn=lambda: model_fn("EfficientNetB1", input_shape=INPUT_SHAPE, num_classes=NUM_CLASSES),
    trainDataset=trainDataset,
    valDataset=valDataset,
    testDataset=testDataset,
    epochs=epochs,
    class_labels = class_labels
)

In [ ]:
# 📊 Plot Accuracy of EfficientNetB1
plot_accuracy("EfficientNetB1_model.keras")

# 📉 Plot Loss of EfficientNetB1
plot_loss("EfficientNetB1_model.keras")

In [ ]:
# 🔧 Set model name and epochs for EfficientNetB2
model_name = "EfficientNetB2_model.keras"
epochs = EPOCHS
class_labels = ["Normal", "Stone"]

# 🚀 Train the EfficientNetB2 model
best_model = train_model(
    model_name=model_name,
    model_fn=lambda: model_fn("EfficientNetB2", input_shape=INPUT_SHAPE, num_classes=NUM_CLASSES),
    trainDataset=trainDataset,
    valDataset=valDataset,
    testDataset=testDataset,
    epochs=epochs,
    class_labels = class_labels
)

In [ ]:
# 📊 Plot Accuracy of EfficientNetB2
plot_accuracy("EfficientNetB2_model.keras")

# 📉 Plot Loss of EfficientNetB2
plot_loss("EfficientNetB2_model.keras")

In [ ]:
# Comparison of the "Simple_Cnn_Model", "AlexNet", "VGGNet", "ResNet34" models [Accuracy & Loss]
compare_models_accuracy(["simple_cnn_model.keras", "AlexNet_model.keras", "VGGNet_model.keras", "ResNet34_model.keras"])
compare_models_loss(["simple_cnn_model.keras", "AlexNet_model.keras", "VGGNet_model.keras", "ResNet34_model.keras"])

In [ ]:
# Comparison of the "EfficientNetB0" "EfficientNetB1", "EfficientNetB2" models [Accuracy & Loss]
compare_models_accuracy(["EfficientNetB0_model.keras", "EfficientNetB1_model.keras", "EfficientNetB2_model.keras"])
compare_models_loss(["EfficientNetB0_model.keras", "EfficientNetB1_model.keras", "EfficientNetB2_model.keras"])


**Evaluation of Models & Results**

In [ ]:
def summarize_results(results_list):
    """
    results_list: A list of dictionaries containing information for each model.
    Example:
    results_list = [
        {
            "model_name": "AlexNet",
            "train_time_sec": 152.34,
            "params": {"batch_size": 32, "lr": 0.001, "dropout": 0.5},
            "train_acc": 0.945,
            "val_acc": 0.932,
            "test_acc": 0.928,
            "csv_path": "predictions_alexnet.csv"
        },
        ...
    ]
    """
    summary_data = []
    for res in results_list:
        summary_data.append({
            "Model": res["model_name"],
            "Train Time (min)": round(res["train_time_sec"] / 60, 2),
            "Best Parameters": ", ".join([f"{k}={v}" for k, v in res["params"].items()]),
            "Train Accuracy": f"{res['train_acc']:.3f}",
            "Validation Accuracy": f"{res['val_acc']:.3f}",
            "Test Accuracy": f"{res['test_acc']:.3f}",
            "CSV Predictions": f"[Download]({res['csv_path']})"
        })
    
    results_df = pd.DataFrame(summary_data)
    return results_df

In [ ]:
results_list = [
    {
        "model_name": "AlexNet",
        "train_time_sec": 152.34,
        "params": {"batch_size": 32, "lr": 0.001, "dropout": 0.5},
        "train_acc": 0.985,
        "val_acc": 0.982,
        "test_acc": 0.958,
        "csv_path": "predictions_alexnet.csv"
    },
    {
        "model_name": "VGGNet",
        "train_time_sec": 178.90,
        "params": {"batch_size": 32, "lr": 0.001, "dropout": 0.5},
        "train_acc": 0.980,
        "val_acc": 0.968,
        "test_acc": 0.965,
        "csv_path": "predictions_vgg.csv"
    }
]

##summary_df = summarize_results(results_list)
##summary_df

In [ ]:
# For nice plots
sns.set(style="whitegrid", context="notebook")


# ============================================
# 📌 Step 1: Configuration
#    - Define your model files and the corresponding prediction CSV naming rule.
#    - My CSV filenames follow "<ModelName>_model_predictions.csv".
#    - Adjust "model_files" and "csv_dir" if needed.
# ============================================
model_files = [
    "simple_cnn_model.keras",
    "AlexNet_model.keras",
    "VGGNet_model.keras",
    "ResNet34_model.keras",
]

# Where the prediction CSVs were saved during training (often /kaggle/working)
csv_dir = "/kaggle/working"

# Class labels (order matters). We will derive positive-class index from the generator.
class_labels = ["Normal", "Stone"]  # Example: "Stone" is the positive class
positive_class_name = "Stone"


# ============================================
# 📌 Step 2: Ground truth (y_true) from labeled test generator
#    - We created this "testDataset_with_labels" earlier with shuffle=False.
#    - We use `.classes` to get numeric labels aligned with the input order.
#    - We also find which index corresponds to the positive class (e.g., "Stone").
# ============================================
y_true_numeric = testDataset_with_labels.classes  # integers per Keras class_indices
class_indices = testDataset_with_labels.class_indices  # dict: class_name -> index
# Positive class index (e.g., Stone -> 1). If missing, default to 1.
pos_index = class_indices.get(positive_class_name, 1)

# Convert y_true to binary {0,1} where 1 = positive class
y_true_numeric = np.array(testDataset_with_labels.classes)  
y_true_bin = (y_true_numeric == pos_index).astype(int)

# Inverse mapping: index -> class_name
inv_class_indices = {v: k for k, v in class_indices.items()}
y_true_names = np.array([inv_class_indices[i] for i in y_true_numeric])


# ============================================
# 📌 Step 3: Utilities to load predictions from CSVs
#    - We support both "hard labels only" and "probabilities + labels".
#    - Expected CSV columns (any of these patterns):
#        a) "label" (0/1 or class names)  -> hard predictions only
#        b) Probability columns:
#           - one column per class, named exactly like class_labels, e.g. "Normal","Stone"
#           - or prefixed names like "prob_Normal","prob_Stone"
#           - or generic names like "p0","p1" or "prob_0","prob_1"
#    - The function returns:
#        - y_pred_hard: np.ndarray int {0,1} (always)
#        - y_proba_pos: np.ndarray float in [0,1] for positive class (if available, else None)
# ============================================
def load_predictions_csv(csv_path, class_labels, pos_index):
    df = pd.read_csv(csv_path)

    # 1) Hard labels
    if "label" not in df.columns:
        raise ValueError(f"'label' column not found in predictions CSV: {csv_path}")

    # If 'label' is string class names -> map to indices using class_labels order
    if df["label"].dtype == object:
        # Build name->index map from class_labels
        name_to_idx = {name: i for i, name in enumerate(class_labels)}
        y_pred_hard = df["label"].map(name_to_idx).values
    else:
        # Already numeric; assume they match your Keras indices
        y_pred_hard = df["label"].values.astype(int)

    # 2) Try to detect probability columns for classes
    #    We try multiple patterns to be robust.
    proba_columns_candidates = [
        class_labels,  # e.g., ["Normal","Stone"]
        [f"prob_{c}" for c in class_labels],  # e.g., ["prob_Normal","prob_Stone"]
        [f"p{i}" for i in range(len(class_labels))],  # e.g., ["p0","p1"]
        [f"prob_{i}" for i in range(len(class_labels))],  # e.g., ["prob_0","prob_1"]
    ]

    y_proba_pos = None
    for cols in proba_columns_candidates:
        if all(col in df.columns for col in cols):
            # We found a set of probability columns that matches number of classes
            proba_matrix = df[cols].values.astype(float)
            # Extract the positive class probability column
            y_proba_pos = proba_matrix[:, pos_index]
            break

    return y_pred_hard, y_proba_pos


# ============================================
# 📌 Step 4: Evaluate a single model's predictions
#    - Always computes: Accuracy, Precision, Recall, F1 (binary)
#    - If y_proba_pos exists: computes ROC-AUC and log_loss as well
# ============================================
def evaluate_predictions(y_true_bin, y_pred_hard, y_proba_pos, model_name):
    # Core metrics from hard predictions
    acc = accuracy_score(y_true_bin, y_pred_hard)
    prec = precision_score(y_true_bin, y_pred_hard, zero_division=0)
    rec = recall_score(y_true_bin, y_pred_hard, zero_division=0)
    f1 = f1_score(y_true_bin, y_pred_hard, zero_division=0)

    metrics = {
        "Model": model_name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1": f1,
        "ROC_AUC": None,
        "LogLoss": None,
    }

    # If we have probabilities, add ROC-AUC and log-loss
    if y_proba_pos is not None:
        # ROC-AUC: needs positive-class probabilities
        try:
            roc = roc_auc_score(y_true_bin, y_proba_pos)
            metrics["ROC_AUC"] = roc
        except Exception as e:
            metrics["ROC_AUC"] = None

        # LogLoss: for binary, we can pass a 2D prob array [[p0, p1], ...]
        try:
            proba_2d = np.column_stack([1.0 - y_proba_pos, y_proba_pos])
            ll = log_loss(y_true_bin, proba_2d, labels=[0, 1])
            metrics["LogLoss"] = ll
        except Exception as e:
            metrics["LogLoss"] = None

    return metrics


# ============================================
# 📌 Step 5: Load all models' predictions from CSV and evaluate
#    - We derive CSV filename for each model by replacing "_model.keras" -> "_model_predictions.csv".
#    - Builds a results table + a dict of prediction arrays for ensembling.
# ============================================
all_results = []
hard_preds_dict = {}   # model_name -> y_pred_hard (for hard voting)
proba_preds_dict = {}  # model_name -> y_proba_pos (for soft voting, if available)

for model_file in model_files:
    model_name = model_file.replace(".keras", "")
    csv_name = model_name + "_predictions.csv"
    csv_path = os.path.join(csv_dir, csv_name)

    if not os.path.exists(csv_path):
        print(f"⚠️ CSV not found for {model_name}: {csv_path}. Skipping.")
        continue

    # Load predictions for this model
    y_pred_hard, y_proba_pos = load_predictions_csv(csv_path, class_labels, pos_index)

    # Store for ensembling
    hard_preds_dict[model_name] = y_pred_hard
    if y_proba_pos is not None:
        proba_preds_dict[model_name] = y_proba_pos

    # Evaluate and collect metrics
    res = evaluate_predictions(y_true_bin, y_pred_hard, y_proba_pos, model_name)
    all_results.append(res)

# Build a tidy DataFrame
results_df = pd.DataFrame(all_results)

# Sort by Accuracy (desc) for presentation
if not results_df.empty:
    results_df = results_df.sort_values(by="Accuracy", ascending=False).reset_index(drop=True)

print("✅ Individual model results:")
display(results_df)


# ============================================
# 📌 Step 6: Ensembling
#    A) Hard voting: average of hard labels (0/1) and round → robust when no probabilities are available.
#    B) Soft voting: average of positive-class probabilities (requires all models to have probabilities).
# ============================================
# ---- A) Hard voting ensemble (works with hard labels only)
if len(hard_preds_dict) >= 2:
    # Stack shape: (num_models, num_samples) -> average across models -> round to {0,1}
    hard_stack = np.vstack([arr for arr in hard_preds_dict.values()])
    ensemble_hard = np.round(hard_stack.mean(axis=0)).astype(int)

    hard_metrics = evaluate_predictions(y_true_bin, ensemble_hard, None, "Ensemble (Hard Voting)")
    print("\n✅ Ensemble (Hard Voting) results:")
    display(pd.DataFrame([hard_metrics]))

    # Append to results_df for plotting
    results_df = pd.concat([results_df, pd.DataFrame([hard_metrics])], ignore_index=True)

# ---- B) Soft voting ensemble (requires probabilities for ALL models used)
if len(proba_preds_dict) == len(hard_preds_dict) and len(proba_preds_dict) >= 2:
    # Average probabilities across models, then threshold at 0.5 for hard class
    proba_stack = np.vstack([arr for arr in proba_preds_dict.values()])
    ensemble_proba = proba_stack.mean(axis=0)  # averaged positive-class probability
    ensemble_soft = (ensemble_proba >= 0.5).astype(int)

    soft_metrics = evaluate_predictions(y_true_bin, ensemble_soft, ensemble_proba, "Ensemble (Soft Voting)")
    print("\n✅ Ensemble (Soft Voting) results:")
    display(pd.DataFrame([soft_metrics]))

    # Append to results_df for plotting
    results_df = pd.concat([results_df, pd.DataFrame([soft_metrics])], ignore_index=True)
else:
    print("\nℹ️ Soft voting ensemble skipped (probabilities not available for all models).")


# ============================================
# 📌 Step 7: Confusion matrices & classification reports
#    - Example: for the top-1 model by Accuracy and for ensembles (if computed)
# ============================================
def plot_conf_mat_and_report(y_true_bin, y_pred_hard, title):
    cm = confusion_matrix(y_true_bin, y_pred_hard, labels=[0, 1])
    plt.figure(figsize=(4.5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
                xticklabels=["Normal", positive_class_name],
                yticklabels=["Normal", positive_class_name])
    plt.title(f"Confusion Matrix - {title}")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.tight_layout()
    plt.show()

    print(f"📋 Classification Report - {title}")
    print(classification_report(y_true_bin, y_pred_hard, target_names=["Normal", positive_class_name], digits=4))

# Plot for best single model
if not results_df.empty:
    # pick best by Accuracy
    best_row = results_df.sort_values("Accuracy", ascending=False).iloc[0]
    best_name = best_row["Model"]

    if best_name in hard_preds_dict:
        plot_conf_mat_and_report(y_true_bin, hard_preds_dict[best_name], f"{best_name} (Best Single)")

# Plot for top-3-models (simple_cnn_model, AlexNet_model, VGGNet_mode)
print("📈 Confusion matrices & classification reports for ⚡ Top-3-Models (simple_cnn_model, AlexNet_model, VGGNet_mode)")
top3_models = ["simple_cnn_model", "AlexNet_model", "VGGNet_model"]

for m in top3_models:
    if m in hard_preds_dict:
        plot_conf_mat_and_report(y_true_bin, hard_preds_dict[m], m)
    else:
        print(f"⚠️ Predictions not found for {m}")

# Plot for ensembles (if computed)
print("📈 Confusion matrices & classification reports for ⚡ Ensembles (Hard Voting and Soft Voting")
if "Ensemble (Hard Voting)" in results_df["Model"].values:
    plot_conf_mat_and_report(y_true_bin, ensemble_hard, "Ensemble (Hard Voting)")

if "Ensemble (Soft Voting)" in results_df["Model"].values:
    plot_conf_mat_and_report(y_true_bin, ensemble_soft, "Ensemble (Soft Voting)")


# ============================================
# 📌 Step 8: Pretty comparison plots
#    - Bar charts for Accuracy and F1 and Recall
#    - If ROC_AUC is available for at least one model, plot it too
# ============================================
def plot_metric_bar(df, metric, title, ylabel, save_path=None):
    if metric not in df.columns:
        return
    plt.figure(figsize=(8, 5))
    # Order as they appear in df
    sns.barplot(x="Model", y=metric, data=df, edgecolor="black")
    plt.title(title, fontsize=14, fontweight="bold")
    plt.ylabel(ylabel, fontsize=12)
    plt.xlabel("Models", fontsize=12)
    plt.xticks(rotation=20, ha="right", fontsize=11)
    plt.yticks(fontsize=11)
    plt.grid(axis="y", linestyle="--", alpha=0.5)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()

# Order columns and format for display
display_cols = ["Model", "Accuracy", "Precision", "Recall", "F1", "ROC_AUC", "LogLoss"]
results_df = results_df.reindex(columns=[c for c in display_cols if c in results_df.columns])
print("\n📊 Final summary table:")
display(results_df.style.format(precision=4))

# Plots
plot_metric_bar(results_df, "Accuracy", "Model Accuracy Comparison", "Accuracy", save_path="compare_accuracy.png")
plot_metric_bar(results_df, "F1", "Model F1-Score Comparison", "F1-Score", save_path="compare_f1.png")
plot_metric_bar(results_df, "Recall", "Model Recall Comparison", "Recall-Score", save_path="compare_recall.png")
if results_df["ROC_AUC"].notna().any():
    plot_metric_bar(results_df, "ROC_AUC", "Model ROC-AUC Comparison", "ROC-AUC", save_path="compare_rocauc.png")


# ============================================
# 📌 Step 9: Save the final summary
# ============================================
# 1) final_results_summary.csv: contains the results of the current run, organized and formatted for graphing.
results_df.to_csv("final_results_summary.csv", index=False)
print("\n💾 Saved: final_results_summary.csv")


# ============================================
# 📌 Step 10: Save metrics for each model into central log file
#    - Uses your function save_model_metrics()
#    - This ensures results persist across different runs / versions
# ============================================
# 2) models_evaluation_results.csv: a central file where new records will be added with each subsequent run, including all models and all versions.
for _, row in results_df.iterrows():
    model_name = row["Model"]

    # We already have evaluation results in row, no need to recompute predictions again
    # Just repackage them into save_model_metrics format
    result_row = {
        "Model": model_name,
        "Accuracy": row.get("Accuracy"),
        "Precision": row.get("Precision"),
        "Recall": row.get("Recall"),
        "F1-score": row.get("F1"),
        "LogLoss": row.get("LogLoss")
    }

    # Append to models_evaluation_results.csv (global log file)
    if os.path.exists("models_evaluation_results.csv"):
        df_old = pd.read_csv("models_evaluation_results.csv")
        df_new = pd.concat([df_old, pd.DataFrame([result_row])], ignore_index=True)
    else:
        df_new = pd.DataFrame([result_row])

    df_new.to_csv("models_evaluation_results.csv", index=False)

print("\n💾 Saved: models_evaluation_results.csv (central log of all runs)")

In [ ]:
def collect_results(model_files, valDataset, testDataset, df_test, class_labels=["Normal", "Stone"]):
    results_list = []
    
    # Number of steps to test
    test_steps = int(np.ceil(len(df_test) / testDataset.batch_size))
    
    for model_file in model_files:
        model_name = model_file.replace("_model.keras", "")
        
        # --- Load Meta Info ---
        meta_path = model_file.replace(".keras", "_meta.json")
        if os.path.exists(meta_path):
            with open(meta_path, "r") as f:
                meta_info = json.load(f)
            train_time_sec = meta_info.get("train_time_sec", None)
            params = meta_info.get("params", {})
        else:
            train_time_sec = None
            params = {}
        
        # --- Load Training History ---
        hist_path = model_file.replace(".keras", "_history.npz")
        if os.path.exists(hist_path):
            hist_data = np.load(hist_path, allow_pickle=True)
            val_acc_final = hist_data["val_acc"][-1] if "val_acc" in hist_data else None
            train_acc_final = hist_data["acc"][-1] if "acc" in hist_data else None
        else:
            val_acc_final = None
            train_acc_final = None
        
        # --- Predict on Test Dataset (محدود کردن تعداد steps) ---
        model = tf.keras.models.load_model(model_file)
        preds = model.predict(testDataset, steps=test_steps, verbose=0)
        y_pred = np.argmax(preds, axis=1)
        
        # --- Test Accuracy ---
        test_acc = None
        if "label" in df_test.columns:
            y_true = df_test["label"].map({cls: i for i, cls in enumerate(class_labels)}).values
            test_acc = np.mean(y_true == y_pred)
        
        # --- CSV Predictions Path ---
        csv_path = model_file.replace(".keras", "_test_predictions.csv")
        
        results_list.append({
            "Model": model_name,
            "Train Time (min)": round(train_time_sec / 60, 2) if train_time_sec else None,
            "Best Parameters": ", ".join([f"{k}={v}" for k, v in params.items()]) if params else None,
            "Train Accuracy": round(train_acc_final, 4) if train_acc_final else None,
            "Validation Accuracy": round(val_acc_final, 4) if val_acc_final else None,
            "Test Accuracy": round(test_acc, 4) if test_acc else None,
            "CSV Predictions": csv_path if os.path.exists(csv_path) else None
        })
    
    results_df = pd.DataFrame(results_list)
    return results_df

In [ ]:
final_summary_df = collect_results(model_files, valDataset, testDataset, df_test)
print(final_summary_df)